In [38]:
import pandas as pd

master_data_csv = r"C:\Users\rajne\Desktop\trains_schedule_csv_ToUse.csv"

In [39]:
trains_schedule_df = pd.read_csv(master_data_csv)      # core df made above using schedule.json file @ cell 11

trains_schedule_df['departure_time'] = trains_schedule_df['departure_time'].replace(['None',None],'00:00:00')


In [40]:
trains_schedule_df = trains_schedule_df[trains_schedule_df['departure_time'] != 'BJP']
trains_schedule_df = trains_schedule_df[trains_schedule_df['departure_time'] != 'UBL']
trains_schedule_df = trains_schedule_df[trains_schedule_df['departure_time'] != 'SUR']
trains_schedule_df = trains_schedule_df[trains_schedule_df['departure_time'] != 'GDG']

trains_schedule_df.dropna(subset = ['train_num','train_name'], inplace= True)

trains_groups_obj = trains_schedule_df.groupby('train_num')

trains_groups_obj.size()

train_num
10103    20
10104    20
1011     18
10111    20
10112    20
         ..
99904    12
99905    11
99906    12
99907    12
99908    12
Length: 11112, dtype: int64

In [41]:
trains_schedule_df

,train_num,train_name,station_code,station_name,departure_time
0,107,SWV-MAO-VLNK,SWV,SAWANTWADI R,10:25:00
1,107,SWV-MAO-VLNK,THVM,THIVIM,11:08:00
2,107,SWV-MAO-VLNK,KRMI,KARMALI,11:30:00
3,107,SWV-MAO-VLNK,MAO,MADGOAN JN.,00:00:00
4,108,VLNK-MAO-SWV,MAO,MADGOAN JN.,20:30:00
...,...,...,...,...,...
186119,99908,EMU,AKRD,AKURDI,23:31:00
186120,99908,EMU,DEHR,DEHU ROAD,23:36:00
186121,99908,EMU,BGWI,BEGDAEWAI,23:40:00
186122,99908,EMU,GRWD,GHORAWADI,23:42:00


In [42]:
# to evaluvate journey day for the stop sequence of the train

from datetime import datetime, time

def time_conversion(time_str_row):
    if time_str_row in ['BJP' , 'UBL', 'SUR', 'GDG']:
        return '00:00:00'
    formatted_time = datetime.strptime(time_str_row, "%H:%M:%S").time()
    return formatted_time

# ---------------------------------------------------------------------------------

for train_num, mini_df in trains_groups_obj:
    # mini_df = mini_df.sort_values('departure_time')
    mini_df['updated_time'] = mini_df['departure_time'].apply(time_conversion)

    anchor_time = mini_df['updated_time'].iloc[0]  # optional method: mini_df.loc[mini_df.index[0], 'updated_time']
    # print(f"anchor time: {anchor_time}")
    default_day = 1

    time_series = mini_df['updated_time']
    
    for indx in mini_df.index:
        
        time = time_series.loc[indx]
        
        # print(f"time: {time}")
        if time < anchor_time:
            default_day += 1
            # print('Day change or Wrong time: Check!')
            
        trains_schedule_df.at[indx, 'day'] = default_day
        anchor_time = time

    # print(mini_df)

In [43]:
trains_schedule_df

,train_num,train_name,station_code,station_name,departure_time,day
0,107,SWV-MAO-VLNK,SWV,SAWANTWADI R,10:25:00,1.0
1,107,SWV-MAO-VLNK,THVM,THIVIM,11:08:00,1.0
2,107,SWV-MAO-VLNK,KRMI,KARMALI,11:30:00,1.0
3,107,SWV-MAO-VLNK,MAO,MADGOAN JN.,00:00:00,2.0
4,108,VLNK-MAO-SWV,MAO,MADGOAN JN.,20:30:00,1.0
...,...,...,...,...,...,...
186119,99908,EMU,AKRD,AKURDI,23:31:00,1.0
186120,99908,EMU,DEHR,DEHU ROAD,23:36:00,1.0
186121,99908,EMU,BGWI,BEGDAEWAI,23:40:00,1.0
186122,99908,EMU,GRWD,GHORAWADI,23:42:00,1.0


In [44]:
time_transformation = pd.to_timedelta(trains_schedule_df['departure_time'])

trains_schedule_df['timeline'] = time_transformation + pd.to_timedelta(trains_schedule_df['day'], unit = 'D')

tains_cdf = trains_schedule_df.copy()
sorted_df = tains_cdf.sort_values(['train_num','timeline'])
sorted_df

,train_num,train_name,station_code,station_name,departure_time,day,timeline
4354,10103,MANDOVI EXPR,CSMT,CST-MUMBAI,07:10:00,1.0,1 days 07:10:00
4355,10103,MANDOVI EXPR,DR,DADAR,07:25:00,1.0,1 days 07:25:00
4356,10103,MANDOVI EXPR,TNA,THANE,07:50:00,1.0,1 days 07:50:00
4357,10103,MANDOVI EXPR,PNVL,PANVEL,08:30:00,1.0,1 days 08:30:00
4358,10103,MANDOVI EXPR,MNI,MANGAON,10:35:00,1.0,1 days 10:35:00
...,...,...,...,...,...,...,...
186119,99908,EMU,AKRD,AKURDI,23:31:00,1.0,1 days 23:31:00
186120,99908,EMU,DEHR,DEHU ROAD,23:36:00,1.0,1 days 23:36:00
186121,99908,EMU,BGWI,BEGDAEWAI,23:40:00,1.0,1 days 23:40:00
186122,99908,EMU,GRWD,GHORAWADI,23:42:00,1.0,1 days 23:42:00


In [46]:
sorted_df['stop_sequence'] = sorted_df.groupby('train_num').cumcount()+1
sorted_df

,train_num,train_name,station_code,station_name,departure_time,day,timeline,stop_sequence
4354,10103,MANDOVI EXPR,CSMT,CST-MUMBAI,07:10:00,1.0,1 days 07:10:00,1
4355,10103,MANDOVI EXPR,DR,DADAR,07:25:00,1.0,1 days 07:25:00,2
4356,10103,MANDOVI EXPR,TNA,THANE,07:50:00,1.0,1 days 07:50:00,3
4357,10103,MANDOVI EXPR,PNVL,PANVEL,08:30:00,1.0,1 days 08:30:00,4
4358,10103,MANDOVI EXPR,MNI,MANGAON,10:35:00,1.0,1 days 10:35:00,5
...,...,...,...,...,...,...,...,...
186119,99908,EMU,AKRD,AKURDI,23:31:00,1.0,1 days 23:31:00,8
186120,99908,EMU,DEHR,DEHU ROAD,23:36:00,1.0,1 days 23:36:00,9
186121,99908,EMU,BGWI,BEGDAEWAI,23:40:00,1.0,1 days 23:40:00,10
186122,99908,EMU,GRWD,GHORAWADI,23:42:00,1.0,1 days 23:42:00,11


In [48]:
grouped_trains = sorted_df.groupby('train_num')
grouped_trains.size()

train_num
10103    20
10104    20
1011     18
10111    20
10112    20
         ..
99904    12
99905    11
99906    12
99907    12
99908    12
Length: 11112, dtype: int64

In [49]:
grouped_trains.get_group('70101')

,train_num,train_name,station_code,station_name,departure_time,day,timeline,stop_sequence
162359,70101,PERN - MAO-,PERN,PERNEM,17:35:00,1.0,1 days 17:35:00,1
162360,70101,PERN - MAO-,THVM,THIVIM,17:48:00,1.0,1 days 17:48:00,2
162361,70101,PERN - MAO-,KRMI,KARMALI,18:08:00,1.0,1 days 18:08:00,3
162362,70101,PERN - MAO-,VEN,VERNA,18:28:00,1.0,1 days 18:28:00,4
162363,70101,PERN - MAO-,MJO,MAJORDE JN.,18:41:00,1.0,1 days 18:41:00,5
162364,70101,PERN - MAO-,SRVX,SURAVALI (H),18:47:00,1.0,1 days 18:47:00,6
162365,70101,PERN - MAO-,MAO,MADGOAN JN.,19:10:00,1.0,1 days 19:10:00,7
162366,70101,PERN - MAO-,BLLI,BALLI,19:32:00,1.0,1 days 19:32:00,8
162367,70101,PERN - MAO-,CNO,CANACONA,19:50:00,1.0,1 days 19:50:00,9
162368,70101,PERN - MAO-,LOL,LOLIEM,20:02:00,1.0,1 days 20:02:00,10


In [51]:
grouped_trains.get_group('10112')

,train_num,train_name,station_code,station_name,departure_time,day,timeline,stop_sequence
4414,10112,KONKAN KANYA,MAO,MADGOAN JN.,18:00:00,1.0,1 days 18:00:00,1
4415,10112,KONKAN KANYA,KRMI,KARMALI,18:32:00,1.0,1 days 18:32:00,2
4416,10112,KONKAN KANYA,THVM,THIVIM,18:55:00,1.0,1 days 18:55:00,3
4417,10112,KONKAN KANYA,PERN,PERNEM,19:11:00,1.0,1 days 19:11:00,4
4418,10112,KONKAN KANYA,SWV,SAWANTWADI R,19:37:00,1.0,1 days 19:37:00,5
4419,10112,KONKAN KANYA,KUDL,KUDAL,20:10:00,1.0,1 days 20:10:00,6
4420,10112,KONKAN KANYA,SNDD,SINDHU DURG,20:30:00,1.0,1 days 20:30:00,7
4421,10112,KONKAN KANYA,KKW,KANKAVALI,21:10:00,1.0,1 days 21:10:00,8
4422,10112,KONKAN KANYA,VBW,VAIBHAVWADI,21:40:00,1.0,1 days 21:40:00,9
4423,10112,KONKAN KANYA,RAJP,RAJAPUR ROAD,21:54:00,1.0,1 days 21:54:00,10


In [52]:
# saving transformed file , ready to ingest in database using sqlite
to_path = r"D:\01_Research_and_Development\PROJECTS\railink\prepared_schedules_data.csv"

# Using pandas in-built function 'to_csv'
#    'index=False' prevents Pandas from saving its arbitrary loop row tracking numbers as an extra unnamed column!
sorted_df.to_csv(to_path, index=False, encoding='utf-8')


----
----
----

# DATABASE CREATION AND DATA INGESTION FOT RAILINK_webapp:

In [53]:
# Creating DB and tables! (EXECUTED CODE! MARKED AS RAW CELL TO AVOID RERUN) < < < < < < < < 

import sqlite3

with sqlite3.connect("railways_hop_network.db") as connection:
    cursor = connection.cursor()

    cursor.execute("""CREATE TABLE IF NOT EXISTS schedules(
                    train_num INTEGER NOT NULL,
                    train_name TEXT NOT NULL,
                    station_code TEXT NOT NULL,
                    station_name TEXT NOT NULL,
                    departure_time TEXT NOT NULL,
                    day INTEGER NOT NULL,
                    timeline TEXT NOT NULL,
                    stop_sequence INTEGER NOT NULL
                    )""")
    print("Schedules table created successfully")

    connection.commit()

Schedules table created successfully


In [58]:
# Reading schedules csv and inserting data to table! : > > [Marked Raw cell to avoid rerun and data dupliaction] < < < < <
import csv

schedules_file = r"D:\01_Research_and_Development\PROJECTS\railink\prepared_schedules_data.csv"

with sqlite3.connect("railways_hop_network.db") as connection:
    cursor = connection.cursor()

    schedules_data_as_tupls = []
    
    with open(schedules_file,'r', newline='') as sf:
        csv_reads = csv.DictReader(sf)

        for row in csv_reads:
            train_num = row['train_num']
            train_name	= row['train_name']
            station_code = row['station_code']
            station_name = row['station_name']
            departure_time	= row['departure_time']
            day	= row['day']
            timeline = row['timeline']
            stop_sequence = row['stop_sequence']

            schedules_data_as_tupls.append((train_num,train_name,station_code,station_name, departure_time,day,timeline,stop_sequence))

    print("Schedules CSV data loaded for ingestion...")
    cursor.executemany("""INSERT INTO schedules VALUES(?,?,?,?,?,?,?,?)""", schedules_data_as_tupls)
    print("Data Ingestion Successfull!")
    connection.commit()


Schedules CSV data loaded for ingestion...
Data Ingestion Successfull!


In [60]:
with sqlite3.connect("railways_hop_network.db") as connection:

    cursor = connection.cursor()

    cursor.execute("SELECT * FROM schedules LIMIT 15")

    fetched = cursor.fetchall()

for row in fetched:
    print(row)

(10103, 'MANDOVI EXPR', 'CSMT', 'CST-MUMBAI', '07:10:00', 1, '1 days 07:10:00', 1)
(10103, 'MANDOVI EXPR', 'DR', 'DADAR', '07:25:00', 1, '1 days 07:25:00', 2)
(10103, 'MANDOVI EXPR', 'TNA', 'THANE', '07:50:00', 1, '1 days 07:50:00', 3)
(10103, 'MANDOVI EXPR', 'PNVL', 'PANVEL', '08:30:00', 1, '1 days 08:30:00', 4)
(10103, 'MANDOVI EXPR', 'MNI', 'MANGAON', '10:35:00', 1, '1 days 10:35:00', 5)
(10103, 'MANDOVI EXPR', 'KHED', 'KHED', '11:27:00', 1, '1 days 11:27:00', 6)
(10103, 'MANDOVI EXPR', 'CHI', 'CHIPLUN', '11:59:00', 1, '1 days 11:59:00', 7)
(10103, 'MANDOVI EXPR', 'SGR', 'SANGMESHWAR', '12:36:00', 1, '1 days 12:36:00', 8)
(10103, 'MANDOVI EXPR', 'RN', 'RATNAGIRI', '13:20:00', 1, '1 days 13:20:00', 9)
(10103, 'MANDOVI EXPR', 'ADVI', 'ADAVALI', '13:55:00', 1, '1 days 13:55:00', 10)
(10103, 'MANDOVI EXPR', 'RAJP', 'RAJAPUR ROAD', '14:31:00', 1, '1 days 14:31:00', 11)
(10103, 'MANDOVI EXPR', 'VBW', 'VAIBHAVWADI', '14:47:00', 1, '1 days 14:47:00', 12)
(10103, 'MANDOVI EXPR', 'KKW', 'KANK